# Analisis de Finanzas y Riesgo Crediticio

Proyecto: Banca Atlas

Equip_30

Equipo: Finanzas y Riesgo Crediticio 

Mariia Zaitseva

_____

Semana: 1

_____

**En qué medida los clientes con saldos más bajos están
en mayor riesgo de incumplimiento de crédito, y cómo debemos ajustar nuestras políticas de crédito para
mitigar ese riesgo?**

### Imports

In [1]:
import pandas as pd
import numpy as np

# graficos
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

# DarkMode Plotly
pio.templates.default = "plotly_dark"

pio.templates["custom"] = pio.templates["plotly_dark"]
pio.templates["custom"].layout.paper_bgcolor = "#050a30"
pio.templates["custom"].layout.plot_bgcolor  = "#050a30"
pio.templates.default = "custom"

# modelo
from scipy.stats.contingency import odds_ratio
from scipy.stats import fisher_exact
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve,  precision_recall_curve, roc_auc_score
from sklearn.metrics import f1_score, classification_report

### Carga de datos

In [2]:
df = pd.read_csv("../Data/05-25-2026/Bank_Marketing_Clean-25_May.csv", index_col=False, encoding='utf-8')

In [3]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10637,10638,33,technician,married,secondary,0,218,1,1,telephone,2,mar,169,4,-1,0,no_pcampaign,0
10638,10639,42,management,single,tertiary,0,1146,1,0,unknown,15,may,98,2,-1,0,no_pcampaign,0
10639,10640,31,unemployed,single,secondary,0,167,0,0,cellular,20,nov,316,1,-1,0,no_pcampaign,0
10640,10641,30,blue-collar,single,secondary,1,447,0,0,cellular,19,nov,426,2,189,6,failure,0


### Exploración de los Datos para el analisis

##### El porcentaje de impagos:

Hay pocos casos de pagos atrasados de un préstamo.

In [4]:
df['default'].value_counts(normalize=True) * 100

default
0    98.543507
1     1.456493
Name: proportion, dtype: float64

In [5]:
df_plot = df.copy()
df_plot['default_label'] = df_plot['default'].map({0: 'No', 1: 'Sí'})

fig = px.histogram(
    df_plot, 
    x='default_label', 
    title='La distribution de impagos',
    labels={'default_label': 'Impago (default)', 'count': 'Número de clientes'},
    category_orders={'default_label': ['No', 'Sí']},
    color='default_label',
    color_discrete_sequence=['#636EFA', '#EF553B']
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    xaxis_tickvals=[0, 1],
    xaxis_ticktext=['No', 'Sí'], 
    showlegend=False,
    yaxis=dict(
        title='', 
        tickformat='d' 
    )
)

fig.update_traces(
    texttemplate='%{y}', 
    textposition='outside', 
    textfont=dict(size=12, weight='bold', color='white') 
)

fig.show()

##### La distribución de saldo:

In [6]:
fig = px.histogram(
    df, 
    x='balance', 
    nbins=100, 
    title='La distribución de saldo',
    labels={'balance': 'Saldo (balance), euro', 'count': 'Número de clientes'},
    range_x=[-5000, 82000]
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    yaxis=dict(
        title='Número de clientes', 
        tickformat='d' 
    ),
    xaxis=dict(
        title='Saldo (balance), euro', 
        tickformat='d' 
    ),
    showlegend=False
)

fig.show()

##### Saldo de los clientes sin y con impago:

In [7]:
fig = px.box(
    df_plot, 
    x='default_label', 
    y='balance',
    title='Balance vs Default',
    labels={'default_label': 'Impago (default)', 'balance': 'Euro'},
    category_orders={'default_label': ['No', 'Sí']},
    color='default_label',
    color_discrete_sequence=['#636EFA', '#EF553B']
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'), 
    yaxis_title_font=dict(size=14, weight='bold'), 
    xaxis_tickfont=dict(size=12), 
    yaxis_tickfont=dict(size=12), 
    yaxis=dict(range=[-8500, 85000], 
               tickformat='d'), 
    showlegend=False
)

fig.show()

##### Indicadores estadísticos del saldo en las cuentas de los clientes sin y con impago:

El conjunto de datos es altamente asimétrico en términos de categoría del balance y número de impagos de préstamos (desequilibrio de clases extremo).

In [8]:
df.groupby('default')['balance'].agg(['mean', 'median', 'min', 'max'])

,mean,median,min,max
default,,,,
0,1560.420330,571.0,-3058,81204
1,-74.019355,0.0,-6847,5249


##### Tasa de impagos si hay préstamo personal (Loan vs Default):

In [9]:
pd.crosstab(
    df['loan'],
    df['default'],
    normalize='index'
)

default,0,1
loan,,
0,0.989105,0.010895
1,0.960641,0.039359


##### Tasa de impagos si hay préstamo hipotecario (Housing vs Default):

In [10]:
pd.crosstab(
    df['housing'],
    df['default'],
    normalize='index'
)

default,0,1
housing,,
0,0.986749,0.013251
1,0.983942,0.016058


##### Tasa de impagos en diferentes tipos de empleo (Job vs Default):

In [11]:
job_risk = pd.crosstab(
    df['job'],
    df['default'],
    normalize='index'
)

job_risk

default,0,1
job,,
admin.,0.992963,0.007037
blue-collar,0.978142,0.021858
entrepreneur,0.967532,0.032468
housemaid,0.968872,0.031128
management,0.986481,0.013519
retired,0.993377,0.006623
self-employed,0.981959,0.018041
services,0.992054,0.007946
student,0.997175,0.002825


## El analisis

Dividir a los clientes en grupos según el saldo de su cuenta para determinar el riesgo de impago.

Dado que los impagos de préstamos son muy escasos en todos los datos disponibles, para obtener un número suficiente de impagos para el análisis y resultados estadísticos estables a partir del análisis, fue definido a los clientes con saldos bajos como aquellos que se encuentran en el 25% inferior de la distribución de saldos.

##### Determinar el límite superior de saldo bajo (25%):

In [12]:
q25 = df['balance'].quantile(0.25)

In [13]:
print(f"El límite superior de saldo bajo es {round(q25)} euro")

El límite superior de saldo bajo es 127 euro


In [14]:
df['low_balance'] = np.where(
    df['balance'] <= q25,
    1,
    0
)

##### Número de clientes en el grupo de saldo bajo y otros clientes:

El número de clientes en el grupo de saldo bajo es 2669.

In [15]:
counts = df['low_balance'].value_counts()
counts

low_balance
0    7973
1    2669
Name: count, dtype: int64

##### Proporción de impagos en el grupo de saldo bajo con respecto al resto:

El número de impagos entre el grupo de clientes con saldo bajo es de 127 casos, en comparación con 28 entre los demás clientes.

In [16]:
contingency = pd.crosstab(
    df['low_balance'],
    df['default']
)

contingency

default,0,1
low_balance,,
0,7945,28
1,2542,127


### La probabilidad de incumplimiento

Los clientes con saldos bajos tienen 13,5 veces más probabilidades de incurrir en impago.

In [17]:
default_rates = (
    df.groupby('low_balance')['default']
    .mean()
    .reset_index(name='default_probability')
)

default_rates

,low_balance,default_probability
0,0,0.003512
1,1,0.047583


### Riesgo relativo (RR):

Los clientes con saldos bajos incurren en impago 13,55 veces más frequente.

In [18]:
low_risk = default_rates.loc[
    default_rates['low_balance'] == 1,
    'default_probability'
].values[0]

normal_risk = default_rates.loc[
    default_rates['low_balance'] == 0,
    'default_probability'
].values[0]

relative_risk = low_risk / normal_risk

print(round(relative_risk,2))

13.55


In [19]:
fig = px.bar(
    default_rates, 
    x='low_balance', 
    y='default_probability',
    color='default_probability',
    title='La probabilidad de impago entre los grupos por saldo',
    labels={'low_balance': '', 'default_probability': 'Probabilidad de impago'},
    category_orders={'low_balance': [0, 1]},
    color_continuous_scale=['green', 'red'] 
)

fig.update_layout(
    title_font=dict(size=16, weight='bold'),
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=['Saldo normal', 'Saldo bajo'] 
    ),
    yaxis=dict(
        title='Probabilidad de impago',
        tickformat='.0%' 
    ),
    showlegend=False
)

fig.show()

### Odds Ratio (OR)

Los clientes con saldos bajos aumentan los riesgos de incurrir en impago 14,17 veces.

In [20]:
result = odds_ratio(contingency.values)

print(round(result.statistic, 2))

14.17


**Conclisión:** 
El riesgo relativo (RR) indica que los clientes con saldos bajos tienen aproximadamente 13,5 veces más probabilidades de incumplir sus pagos en comparación con el resto de la población.

Un Odds Ratio (OR) cercano a 14,2 confirma una relación muy sólida. Este resultado es típico en el análisis de eventos poco frecuentes, situaciones en las que el cálculo de riesgo и la probabilidad tienden a dar el mismo resultado

### Prueba de significancia estadística

Debido al marcado desequilibrio de clases en la variable objetivo (los casos de impago son poco frecuentes), varias celdas de la tabla de contingencia contienen valores muy pequeños. Por lo tanto, se utilizó la prueba exacta de Fisher en lugar de la prueba de chi-cuadrado, ya que no se basa en aproximaciones para muestras grandes y proporciona un p-valor exacto para tablas de contingencia de 2×2.

Fisher Exact Test

In [21]:
oddsratio, p = fisher_exact(contingency)

print(p)

1.2516602277523208e-50


**Conclisión:** statistically significant.

## Regresión logística modelo

Elegir las variables:

In [22]:
features = ['low_balance', 'age', 'loan', 'housing', 'education']

Cambiar las variables categoricas a las variables numericas (one-hot encoding):

In [23]:
df_model = pd.get_dummies(
    df[features],
    drop_first=True
)

In [24]:
df_model = df_model.astype(float)

In [25]:
df_model.dtypes

low_balance            float64
age                    float64
loan                   float64
housing                float64
education_secondary    float64
education_tertiary     float64
dtype: object

Comprueba la ausencia de los valores nulos:

In [26]:
df_model.isna().sum()

low_balance            0
age                    0
loan                   0
housing                0
education_secondary    0
education_tertiary     0
dtype: int64

### Factor de inflación de la varianza (Variance Inflation Factor, VIF)

VIF

Factor de Inflación de la Varianza (VIF): una medida estadística utilizada para evaluar el grado de multicolinealidad entre predictores en regresión múltiple. Indica cuánto aumenta la varianza de las estimaciones de los coeficientes de regresión debido a la dependencia lineal entre las variables independientes.

El VIF se aplica durante la preparación de datos y la selección de características. Los analistas lo utilizan para evaluar la estabilidad de los coeficientes, mejorar la interpretabilidad y prevenir el sobreajuste. En el contexto del aprendizaje automático, el VIF ayuda a construir modelos más robustos y garantiza la validez de las inferencias estadísticas.

In [27]:
vif_df = pd.DataFrame()
vif_df["feature"] = df_model.columns
vif_df["VIF"] = [
    variance_inflation_factor(df_model.values, i)
    for i in range(df_model.shape[1])
]

vif_df.sort_values("VIF", ascending=False)

,feature,VIF
1,age,4.371791
4,education_secondary,3.204693
5,education_tertiary,2.215350
3,housing,1.783667
0,low_balance,1.331164
2,loan,1.166363


Interpretación
Un VIF de 1 indica ausencia de colinealidad. Los valores entre 1 y 5 se consideran aceptables, aunque pueden indicar cierta interdependencia. Un umbral de 10 se utiliza tradicionalmente como señal de alerta de alta multicolinealidad, lo que requiere la revisión del modelo (por ejemplo, eliminar o combinar variables).

**Conclisión:**
No hay multicolinearidad alta. Se utilizan todas las variables para el modelo.

### Entrenar el modelo

Separar el dataset y la variable objetiva:

In [28]:
X = df_model
y = df['default']

Dividir los datos en conjuntos de entrenamiento y prueba:

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Construir el modelo

In [30]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

La probabilidad de que un cliente incumpla con el pago:

In [31]:
y_proba = model.predict_proba(X_test)[:, 1]

El valor umbral es aquel en el que el modelo considera que la probabilidad de impago es suficiente para clasificar a un cliente como poco fiable, es decir, con alta probabilidad de impago. Si el valor es demasiado bajo, muchos clientes se considerarán poco fiables; si es demasiado alto, existe un alto riesgo de conceder un préstamo a un cliente realmente insolvente.

Por defecto, el valor umbral se establece en 0,5. Para mejorar la capacidad predictiva del modelo, se puede intentar ajustar el umbral mediante una búsqueda en cuadrícula discreta.

In [32]:
thresholds = np.arange(0.01, 0.99, 0.01)

best_f1 = -1
best_threshold = 0.5

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

Best threshold: 0.78
Best F1: 0.19753086419753085


**Conclisión:**
El valor umbral (threshold) de clasificación determina el nivel de probabilidad en el que un cliente se clasifica como moroso. Dado que el conjunto de datos está muy desequilibrado, el umbral se optimizó utilizando la puntuación F1, que equilibra la precisión y la exhaustividad y proporciona una métrica de evaluación más informativa que la exactitud.


In [33]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coef': model.coef_[0]
})

coef_df = coef_df.sort_values(
    by='coef',
    ascending=False
)

coef_df

,feature,coef
0,low_balance,2.474568
2,loan,1.072734
3,housing,0.121367
1,age,0.007902
4,education_secondary,-0.230588
5,education_tertiary,-0.576225


Variables claves y coeficientes que influyen en la variable objetivo:

Interpretación:
- Coeficiente (coef) > 0 → aumenta el riesgo de impago
- Coeficiente (coef) < 0 → disminuye el riesgo
- Probabilidad (odds_ratio) > 1 → aumenta la probabilidad de impago
- Probabilidad (odds_ratio) < 1 → disminuye la probabilidad

In [34]:
coef_df['odds_ratio'] = np.exp(coef_df['coef'])
coef_df.sort_values(by='odds_ratio', ascending=False)

,feature,coef,odds_ratio
0,low_balance,2.474568,11.876577
2,loan,1.072734,2.923360
3,housing,0.121367,1.129040
1,age,0.007902,1.007934
4,education_secondary,-0.230588,0.794067
5,education_tertiary,-0.576225,0.562016


**Conclisión:**
El saldo bajo es la señal más importante para predicir la probabilidad del impago.

### Evaluación del modelo

Calcular la probabilidad de impago con el valor umbral (treshold) básico y elegido:

In [35]:
y_pred_base = (y_proba >= 0.5).astype(int)
y_pred_final = (y_proba >= best_threshold).astype(int)

In [36]:
print("Threshold 0.5")
print(classification_report(y_test, y_pred_base))

print(f"\n\nOptimized threshold {best_threshold}")
print(classification_report(y_test, y_pred_final))

Threshold 0.5
              precision    recall  f1-score   support

           0       1.00      0.76      0.86      2098
           1       0.05      0.90      0.10        31

    accuracy                           0.76      2129
   macro avg       0.52      0.83      0.48      2129
weighted avg       0.98      0.76      0.85      2129



Optimized threshold 0.78
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      2098
           1       0.12      0.52      0.20        31

    accuracy                           0.94      2129
   macro avg       0.56      0.73      0.58      2129
weighted avg       0.98      0.94      0.96      2129



##### Calcular la métrica ROC-AUC (Área Bajo la Curva de la Característica Operativa del Receptor)

La métrica ROC-AUC es un indicador que mide la capacidad de discriminación de un modelo de clasificación binaria. 
Muestra la habilidad del modelo para separar las clases positivas de las negativas. El valor del AUC varía entre 0 y 1, donde un valor de 1 representa un modelo perfecto y un valor de 0.5 indica que el modelo no es mejor que una predicción al azar (como lanzar una moneda)

Con un AUC de 0.86, el modelo demuestra una excelente capacidad para distinguir entre clientes cumplidores y clientes en situación de impago (default).

In [37]:
roc_auc = roc_auc_score(y_test, y_proba)
print("ROC-AUC:", round(roc_auc,2))

ROC-AUC: 0.86


##### Dibujar la métrica ROC-AUC 

La curva ROC demuestra que el modelo posee un buen poder discriminatorio, ya que supera significativamente al modelo de referencia aleatorio. El umbral de decisión seleccionado (0,78) corresponde a un punto de operación conservador con una tasa de falsos positivos relativamente baja, priorizando la reducción de predicciones de incumplimiento incorrectas a costa de una menor sensibilidad.

In [38]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

In [39]:
idx = np.argmin(np.abs(thresholds - best_threshold))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr, 
    y=tpr, 
    mode='lines', 
    name='ROC curve',
    line=dict(color='#00CC96', width=2) 
))

fig.add_trace(go.Scatter(
    x=[0, 1], 
    y=[0, 1], 
    mode='lines', 
    name='Adivinanza aleatoria',
    line=dict(dash='dash', color='gray', width=1)
))

fig.add_trace(go.Scatter(
    x=[fpr[idx]], 
    y=[tpr[idx]], 
    mode='markers', 
    name=f'Umbral elegido = {best_threshold:.2f}',
    marker=dict(color='red', size=12) 
))

fig.update_layout(
    title='ROC Curva con el umbral (threshold)',
    title_font=dict(size=16, weight='bold'),
    xaxis_title='Tasa de falsos positivos',
    yaxis_title='Tasa de verdaderos positivos',
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    xaxis=dict(range=[0, 1], zeroline=False),
    yaxis=dict(range=[0, 1], zeroline=False),
    showlegend=True,
    hovermode='closest',

    xaxis_gridcolor='rgba(255,255,255,0.1)',
    yaxis_gridcolor='rgba(255,255,255,0.1)'
)

fig.show()

##### Calcular la métrica Precision–Recall

En escenarios con alta disparidad en la distribución de las clases (datos desbalanceados), la métrica de Accuracy (exactitud global) resulta insuficiente para medir el rendimiento real del modelo. Por este motivo, el análisis se ha centrado en las métricas de Precision (Precisión) и Recall (Exhaustividad).

- **Precision** evalúa la fiabilidad de las predicciones del modelo, determinando la proporción de casos positivos correctamente identificados frente al total de alertas generadas. Un valor alto en esta métrica garantiza la minimización de los falsos positivos.

- **Recall** mide la capacidad del modelo para registrar los eventos de interés, calculando el porcentaje de casos positivos reales que fueron detectados con éxito sobre el total existente. Un valor alto en esta métrica asegura la reducción de los falsos negativos.

Dado que ambos indicadores presentan un comportamiento inversamente proporcional (el incremento de uno suele derivar en la disminución del otro), se ha utilizado el **F1-Score** como métrica de consenso. Al ser la media armónica entre Precision y Recall, el F1-Score proporciona una evaluación robusta y equilibrada del desempeño general del modelo, donde un resultado próximo a 1 valida su eficacia

In [40]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

##### Dibujar la métrica Precision–Recall

La curva de Precision–Recall indica una separabilidad limitada entre las clases de impago y no impago. La precisión disminuye rápidamente a medida que aumenta la exhaustividad, lo que refleja un alto número de falsos positivos al intentar detectar a más personas con impago. El umbral seleccionado (0,78) corresponde a un punto de operación conservador, priorizando la precisión sobre la exhaustividad, pero aún operando en un régimen de baja precisión debido al fuerte desequilibrio de clases.

In [41]:
idx = np.argmin(np.abs(thresholds - best_threshold))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall, 
    y=precision, 
    mode='lines', 
    name='Precision–Recall curve',
    line=dict(color='#EF553B', width=2) 
))

fig.add_trace(go.Scatter(
    x=[recall[idx]], 
    y=[precision[idx]], 
    mode='markers', 
    name=f'Umbral elegido = {best_threshold:.2f}',
    marker=dict(color='red', size=15, line=dict(width=2, color='white')) 
))

fig.update_layout(
    title='Precision–Recall Curva con el umbral',
    title_font=dict(size=16, weight='bold'),
    xaxis_title='Recall',
    yaxis_title='Precision',
    xaxis_title_font=dict(size=14, weight='bold'),
    yaxis_title_font=dict(size=14, weight='bold'),
    xaxis_tickfont=dict(size=12),
    yaxis_tickfont=dict(size=12),
    xaxis=dict(showgrid=True, gridcolor='rgba(255,255,255,0.1)'),
    yaxis=dict(showgrid=True, gridcolor='rgba(255,255,255,0.1)'),
    showlegend=True,
    hovermode='closest',
    plot_bgcolor='#050a30',
    paper_bgcolor='#050a30'
)

fig.show()

## RESUMEN

El modelo calcula la probabilidad de impago para cada cliente.
Para aplicarlo tenemos que definir threshold, el líimite: si el modelo clasifica a un cliente como poco fiable o no. 

Cuanto menor sea este límite, más estricta será la política de préstamos del banco (más "false positive"), lo que significa que se denegarán más préstamos a clientes solventes, pero menores serán los gastos y los beneficios del banco.

Cuanto mayor sea el valor umbral, más clientes podrán obtener un préstamo, pero mayor será el riesgo de impago (menos "false positive"), lo que puede generar mayores gastos, pero también mayores beneficios por los préstamos concedidos y reembolsados. 

#### La probabilidad de incumplimiento: 
Los clientes con saldos bajos tienen **13,5** veces más probabilidades de incurrir en impago.

#### Recomendaciones para la optimización del modelo 

Para mejorar la precisión predictiva del modelo y viabilizar su uso en la toma de decisiones sobre la aprobación de créditos, es necesario incrementar el volumen de datos. Esto incluye tanto ampliar la base general de clientes (más allá de la muestra limitada a los participantes en la campaña de marketing) como incorporar nuevas variables descriptivas por cada usuario, tales como los periodos y montos de los depósitos y retiros en cuenta, así como el historial de pago de créditos anteriores.

#### Recomendaciones para la política de riesgos

Asimismo, respecto a los clientes clasificados con un alto riesgo de default (especialmente aquellos con saldos bajos), se recomienda endurecer la política crediticia mediante las siguientes acciones:

   - Reducir el límite de crédito disponible.

   - Incrementar la tasa de interés aplicable.

   - Disminuir el plazo de amortización del financiamiento.